# ionet cloud docker容器

ionet的基础虚拟机服务，相当于提供了完整的服务器权限，预装了大模型常用的开发环境，当然，如果你想更方便快捷的使用ionet的服务

典型的如快速部署大模型服务，则可以使用ionet提供的docker容器部署服务。

ionet的容器服务链接：https://cloud.io.net/cloud/container

ionet的容器服务文档：https://docs.io.net/docs/deploy-containers

<img src='img/docker1.png' width='600' />

价格方面和虚拟机服务是一样的。

## 基本规则

由于ionet的容器服务主要面向的是基于GPU的大模型推理服务，因此，其使用有一些基本的镜像环境要求：

- 镜像需要支持>=cuda12.8的版本，推荐基于nvidia的镜像(nvidia/cuda:13.0.1-cudnn-devel-ubuntu24.04
或者nvidia/cuda:13.0.1-cudnn-runtime-ubuntu24.04)构建。
- 目前只支持开放一个对外端口，多端口的就需要自己做路由映射了。
- 大模型服务建议基于vllm的镜像构建(vllm/vllm-openai:v0.10.2)

总体上说，ionet的容器服务主要支持的是服务的部署，特别是大模型的快速部署，如果是学习大模型编程，建议还是使用自带juypter的虚拟机服务。

## 例子1：入门例子

我们基于nvidia的镜像，做了个最简单的flask服务，可以用入门测使用：

基本的配置如下：

- Container Image (Public Image): marisming/ionet-python-app:v13.0
- Start Command: 不需要
- port：8000  默认的即可
- Environment Variables：不需要

<img src='img/docker2.png' width='600' />

然后选择服务器，使用4090即可：

<img src='img/docker3.png' width='600' />

时间测试的时候，选择1个小时即可，然后点击部署：

<img src='img/docker4.png' width='600' />

部署后，显示页面如下：

<img src='img/docker5.png' width='600' />


等待部署完成后，就可点击public link链接(样式如：https://385d8a1e02d1-bb4a394c-a0ab-4b7e-bdd0-a23100599ed0-caas.user-hosted-content.io.solutions)，
就能看到服务页面了：

<img src='img/docker6.png' width='600' />

ionet应该是把容器8000端口映射到这个链接了。


## 例子1容器构建代码

具体的容器构建方法如下：

项目文件结构：

```
/nvidia-flask-app
|-- Dockerfile
|-- requirements.txt
`-- app.py
```


文件 1: Dockerfile 
```
# 步骤 1: 使用您提供的 NVIDIA CUDA 13.0.1 官方镜像
FROM nvidia/cuda:13.0.1-cudnn-runtime-ubuntu24.04

# 设置环境变量，避免在构建过程中出现交互式提示
ENV DEBIAN_FRONTEND=noninteractive
ENV CUDA_VERSION=13.0.1
ENV PYTHON_VERSION=3.12

# 步骤 2: 安装 pip
# Ubuntu 24.04 自带 Python 3.12，我们只需要安装 pip
RUN apt-get update && \
    apt-get install -y python3-pip && \
    rm -rf /var/lib/apt/lists/*

# 步骤 3: 设置工作目录
WORKDIR /app

# 步骤 4: 复制并安装 Python 依赖
COPY requirements.txt .
# 使用 --break-system-packages 以兼容 Ubuntu 24.04 的 pip 保护机制
RUN pip install --no-cache-dir --break-system-packages -r requirements.txt

# 步骤 5: 复制你的应用代码
COPY app.py .

# 步骤 6: 暴露应用运行的端口
EXPOSE 8000

# 步骤 7: 定义容器启动时要执行的命令
# Ubuntu 24.04 默认的 python3 就是 3.12
CMD ["python3", "app.py"]

```


文件 2 & 3: requirements.txt 和 app.py
requirements.txt:
```
Flask
```

app.py :

```
from flask import Flask
import os

app = Flask(__name__)

@app.route('/')
def hello_world():
    message = "Hello from a Flask app running in an NVIDIA CUDA 13.0 container!"
    return message

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=8000)
```


然后使用docker命令构建即可：
```
docker build -t marisming/ionet-python-app:v13.0 . #镜像名称写自己的
docker login -u marisming  #写你的用户名
# 输入你们的docker hub密码
docker push marisming/ionet-python-app:v13.0

```

具体文件可见docker/ionet-python-app

## 例子2：自建jupyter

基本的配置如下：

- Container Image (Public Image): marisming/ionet-jupyter:v2
- Start Command: 不需要
- port：8888  
- Environment Variables：不需要

按照这个配置部署ionet的docker容器服务即可。
<img src='img/docker7.png' width='600' />


点击部署完成后点击 public url，默认密码/token为88888888

可以跑下测试代码：

```
import torch

print(f"PyTorch version: {torch.__version__}")

# 检查 CUDA 是否被 PyTorch 检测到
is_available = torch.cuda.is_available()

print(f"Is CUDA available? {is_available}")

# 获取当前默认 GPU 的索引
current_device_index = torch.cuda.current_device()
print(f"Current GPU index: {current_device_index}")


# 获取当前 GPU 的名称
current_device_name = torch.cuda.get_device_name(current_device_index)
print(f"Current GPU name: {current_device_name}")
```


<img src='img/docker8.png' width='600' />

## 例子2 容器构建代码

requirements.txt:

```
# 将 --index-url 替换为 --extra-index-url
# 这会告诉 pip 在默认的 PyPI 仓库之外，也去 PyTorch 的仓库寻找包
--extra-index-url https://download.pytorch.org/whl/cu121

# PyTorch 相关的包会从上面的 extra-index-url 找到
torch
torchvision
torchaudio

# 其他的包会从默认的 PyPI 仓库找到
jupyterlab
pandas
scikit-learn
matplotlib
ipywidgets

```

Dockerfile:
```
# 步骤 1: 使用 NVIDIA CUDA 13.0.1 官方镜像
FROM nvidia/cuda:13.0.1-cudnn-runtime-ubuntu24.04

# 设置环境变量
ENV DEBIAN_FRONTEND=noninteractive

# 步骤 2: 安装 Python pip
RUN apt-get update && \
    apt-get install -y python3-pip && \
    rm -rf /var/lib/apt/lists/*

# 步骤 3: 复制并安装 Python 依赖 (包括 JupyterLab)
WORKDIR /workspace
COPY requirements.txt .
RUN pip install --no-cache-dir --break-system-packages -r requirements.txt

# 步骤 4: 暴露 JupyterLab 默认端口
EXPOSE 8888

# 步骤 5: 设置容器启动命令来运行 JupyterLab
#
# --ip=0.0.0.0      -> 允许从任何IP地址访问 (外网访问的关键)
# --port=8888       -> 指定服务端口
# --no-browser      -> 不要在服务器上尝试打开浏览器
# --allow-root      -> 允许 root 用户运行 (Docker 容器内通常是 root)
# --NotebookApp.token -> 设置一个固定的访问密码/令牌
#
# !!! 安全警告: 请务必将下面的 'your-secret-token-change-me' 替换成一个您自己的、足够复杂的密码 !!!
#
CMD ["jupyter", "lab", "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root", "--NotebookApp.token=88888888"]
```



具体文件可见docker/ionet-jupyter

## 例子3 ： VLLM大模型服务部署
ionet提供docker容器服务的主要目的其实是部署大模型服务

目前使用比较多的就是vLLM：


**vLLM** 是一个开源的、为大型语言模型（LLM）设计的**高性能推理和服务引擎**。

您可以把它理解为一个能让开源大模型（如 Llama, Mistral 等）**跑得更快、更省显存、能同时服务更多人**的加速库。

**核心亮点：**

1.  **极高的性能**：它是目前业界公认的速度最快的推理框架之一。通过`连续批处理(Continuous Batching)`等技术，能最大化利用 GPU 资源，获得极高的吞吐量（Throughput）。

2.  **PagedAttention 技术**：这是 vLLM 的核心创新。它借鉴了操作系统中“虚拟内存”的思想，极大地优化了 LLM 推理中最消耗资源的 KV Cache 显存管理，能将吞吐量提升2-4倍。

3.  **易于使用**：它提供了与 OpenAI API 完全兼容的服务接口。这意味着您只需一行命令就能启动一个高性能的模型服务，并且可以无缝地将现有调用 OpenAI 的代码切换到您自己部署的开源模型上。

**总结一下：**

如果您想在服务器上部署一个开源大模型，并希望它能快速、高效地为大量用户提供服务，**vLLM 是当前最优的解决方案之一**。它是工业级的“引擎”，而不是个人电脑上的“玩具”。

### ionet容器部署vLLM

基于ionet的容器服务，使用vLLM镜像，几分钟就可以上线一个大模型服务，一个典型的例子如下配置：

```
Container Image [Public Image]: vllm/vllm-openai:v0.10.2

Start Command: ["python3", "-m", "vllm.entrypoints.openai.api_server", "--model", "Qwen/Qwen3-8B", "--tensor-parallel-size", "1", "--gpu-memory-utilization", "0.95", "--max-num-seqs", "128", "--async-scheduling", "--enable-prompt-tokens-details", "--tool-call-parser", "openai", "--enable-auto-tool-choice", "--enable-prefix-caching"]

Environment Variable [Normal Variable]: VLLM_RPC_TIMEOUT: 60000

Traffic port : 8000

```

<img src='img/docker9.png' width='600' />


其中命令部分，"--model", "Qwen/Qwen3-8B" 为Huggingface上面的模型，典型的如下链接：

https://docs.vllm.ai/en/stable/models/supported_models.html#text-generation

因为使用的是8B的模型，如果是没有32位的，4090显卡不太够，建议使用H100，一般都是够了的，而且H100做服务肯定是比4090稳定很多。

因为vLLM的是支持兼容openai sdk的，因此，可以改下base_url，即可使用我们部署的服务，

其中base_url就是你部署好后的public url + "/v1/"

测试代码如下：

In [20]:
import openai

IO_URL= "https://f70f31659526-ebf9990b-252a-4173-abd4-05dfa44279ad-caas.user-hosted-content.io.solutions" #public url 写你自己部署的URL

client = openai.OpenAI(
    api_key="EMPTY", #key为空
    base_url=IO_URL + "/v1/",
)

response = client.chat.completions.create(
    model="Qwen/Qwen3-8B", #你部署的模型
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "please introduce chatgpt."},
    ],
    temperature=0.7,
    stream=False,
    max_completion_tokens=5000
)

print(response.choices[0].message.content)

<think>
Okay, the user wants me to introduce ChatGPT. Let me start by explaining what ChatGPT is. It's a large language model developed by OpenAI. I should mention that it's based on the GPT architecture, which stands for Generative Pre-trained Transformer. I need to highlight its capabilities, like generating human-like text, answering questions, and engaging in conversations. Also, it's important to note that it's part of the GPT series, with different versions like GPT-3.5 and GPT-4. I should mention that it's trained on a vast amount of text data, which allows it to understand and generate text across various topics. Maybe include some use cases, such as writing assistance, coding, and customer service. Oh, and don't forget to mention that it's available through OpenAI's platform and can be integrated into different applications. Also, it's crucial to note that while it's powerful, it has limitations and should be used responsibly. Let me structure this in a clear and concise way, 

### 自有模型部署

得益于vLLM的框架特性，加上ionet的容器服务，我们可以方便的部署自有模型，一般只要是Transformer框架的，

模型上传到Huggingface后，都可以简单改下部署时的model名称即可。


这里再测试下我们基于llama-7b模型，使用生物序列持续预训练的模型：dnagpt/llama-gene

部署配置如下：

```
Container Image [Public Image]: vllm/vllm-openai:v0.10.2

Start Command: ["python3", "-m", "vllm.entrypoints.openai.api_server", "--model", "dnagpt/llama-gene", "--tensor-parallel-size", "1", "--gpu-memory-utilization", "0.95", "--max-num-seqs", "128", "--async-scheduling", "--enable-prompt-tokens-details", "--tool-call-parser", "openai", "--enable-auto-tool-choice", "--enable-prefix-caching"]

Environment Variable [Normal Variable]: VLLM_RPC_TIMEOUT: 60000

Traffic port : 8000

```


可以看到，我们只需要修改--model参数即可，因为我们模型是FP16的，因此，7B模型推理需要的显存大致14G左右，因此4090显卡就够了。


<img src='img/docker10.png' width='600' />


然后服务测试如下：

In [22]:
import openai

client = openai.OpenAI(
    api_key="EMPTY",
    base_url="https://bc6dd49019ca-37c85214-71c2-4fb7-a6bd-a2925e8aaa83-caas.user-hosted-content.io.solutions/v1/",
)

response = client.chat.completions.create(
    model="dnagpt/llama-gene",
    messages=[
        {"role": "user", "content": "YRRQMITPRMTIWIKQSIYHDSHSLKAYLGVKQSSTTMYVSGSMTSYFGT"},
    ],
    temperature=0.7,
    stream=False,
    max_completion_tokens=200
)
print(response.choices[0].message.content)

YYRRQMITPRMTIWIKQSIYHDSHSLKAYLGVKQSSTTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFG


### 注意事项

如果模型太大，显存放不下, 部署的时候会出现docker启动错误，在4090部署的时候，可能会出现，

<img src='img/docker11.png' width='600' />


一般的模型，典型的如几十个B，int8量化/FP16，H100都是够了的。


另外，4090启动也会慢不少，如果部署没问题，调用使用时候也需要等几分钟。


另外，ionet的容器服务主要是提供GPU的大模型服务，如果镜像不支持>=cuda12.8, 也是启动不起来的。

<img src='img/docker12.png' width='600' />
